# RAG Workflows with the OpenAI API

This notebook demonstrates a retrieval-augmented generation (RAG) workflow using OpenAI vector stores.

Steps:
1. Load configuration
2. Create or reuse a vector store
3. Upload sample documents
4. Run semantic search
5. Format retrieved context
6. Generate a grounded answer

## 1. Load configuration

In [ ]:
from pathlib import Path
import os
from IPython.display import display, Markdown
from rag_openai_api_workflow.client import make_client
from rag_openai_api_workflow.config import get_settings
from rag_openai_api_workflow.vector_store import create_vector_store
from rag_openai_api_workflow.ingest import ingest_path
from rag_openai_api_workflow.retrieval import search_vector_store, format_search_results
from rag_openai_api_workflow.generation import answer_question
from rag_openai_api_workflow.storage import load_vector_store_id

# Make notebook paths stable whether PyCharm starts Jupyter from the project root
# or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

settings = get_settings()
client = make_client(settings)

PROJECT_ROOT

## 2. Create or reuse a vector store

A vector store holds processed document chunks and embeddings. If there is already a saved vector store ID locally, it is reuse it. Otherwise, a new vector store created.

In [ ]:
vector_store_id = settings.vector_store_id or load_vector_store_id()

if not vector_store_id:
    vector_store_id = create_vector_store(
        client=client,
        name=settings.vector_store_name,
    )

vector_store_id

## 3. Upload sample documents

Now, the sample knowledge-base files are uploaded from `data/sample` and attached to the vector store.

The ingestion code also adds file metadata and static chunking settings.

In [ ]:
RUN_INGESTION = False

In [ ]:
from rag_openai_api_workflow.ingest import ingest_path

sample_docs_path = PROJECT_ROOT / "data" / "sample"

if RUN_INGESTION:
    batch = ingest_path(
        client=client,
        path=sample_docs_path,
        vector_store_id=vector_store_id,
    )
    batch.status
else:
    "Skipped ingestion. Reusing existing vector store."

## 4. Run semantic search

Next, the vector store is searched directly. This retrieves the most relevant chunks for a natural-language query.

In [ ]:
query = "What is the learning format?"

search_results = search_vector_store(
    client=client,
    vector_store_id=vector_store_id,
    query=query,
    max_results=1,
    score_threshold=0.0,
)

len(search_results.data)

## 5. Format retrieved context

The model should receive retrieved chunks in a clear context block. This step formats search results with source numbers, filenames, scores, and retrieved text.

In [ ]:
context = format_search_results(search_results)

display(Markdown("## Retrieved context:"))
display(Markdown(f"```text\n{context}\n```"))

## 6. Generate a grounded answer

Finally, the retrieved context is passed to the model, which is then asked to answer using only that context.

In [ ]:
answer = answer_question(
    client=client,
    model=settings.model,
    question="In 50 words, what is the learning format?",
    search_results=search_results,
)

display(Markdown("## Grounded answer:"))
print(answer)

## Summary

This notebook demonstrates the full RAG workflow:

1. Configuration loading
2. Vector store creation/reuse
3. Document ingestion
4. Semantic search
5. Context formatting
6. Grounded answer generation

The same workflow is also available from the command line through the `rag-openai` CLI.